In [ ]:
from pathlib import Path
from collections import defaultdict
import re

import numpy as np
import pandas as pd
import geopandas as gpd

from eaglei_county_target_processing import build_peak_customers_affected_ge4h_from_exact_outages

from tqdm import tqdm


In [ ]:
eaglei_outage_directory =Path ('/Users/aaronspaulding/Documents/PycharmProjects/outagescraping/EAGLEI/eaglei_outages')
exact_outage_base_directory =Path ('/Users/aaronspaulding/Documents/PycharmProjects/outagescraping/data/Utilities')
county_shapefile_path =Path ('/Users/aaronspaulding/Documents/PycharmProjects/outagescraping/TIGER/tl_2020_us_county/tl_2020_us_county.shp')
aggregated_static_variables_path =Path ('data_cache/aggregated_static_variables.feather')
relevant_storms_path =Path ('data_cache/relevant_storm_tracks.feather')
processed_eaglei_output_directory =Path ('/Users/aaronspaulding/data/processed_eaglei_db')
processed_eaglei_output_directory .mkdir (parents =True ,exist_ok =True )

exact_outage_utilities =[
'Orange and Rockland Utilities Inc',
'Consolidated Edison, Inc',
'Eversource',
'Dominion Energy',
'Public Service Electric and Gas',
'American Electric Power Texas',
]
minimum_duration_hours_for_ge4h_peak =4.0

chunk_size =1_000_000

limit_year_files =None


In [ ]:
relevant_storms =gpd .read_feather (relevant_storms_path )
relevant_storms =relevant_storms [::-1 ].reset_index (drop =True )

county_map_df =pd .read_feather (
aggregated_static_variables_path ,
columns =['FULL_COUNTY_FIPS','county_map'],
)
county_map_df =(
county_map_df
.dropna (subset =['FULL_COUNTY_FIPS','county_map'])
.copy ()
)
county_map_df ['FULL_COUNTY_FIPS']=county_map_df ['FULL_COUNTY_FIPS'].astype (str ).str .zfill (5 )
county_map_df ['county_map']=county_map_df ['county_map'].astype (np .int64 )
county_map_df =(
county_map_df [['FULL_COUNTY_FIPS','county_map']]
.drop_duplicates (subset =['FULL_COUNTY_FIPS'])
.sort_values ('county_map')
.reset_index (drop =True )
)

relevant_storms [['SID','NAME','datetime_min','datetime_max']].head (),county_map_df .head ()


In [ ]:
def to_utc_naive (ts ):
    ts =pd .Timestamp (ts )
    if ts .tzinfo is None :
        return ts
    return ts .tz_convert ('UTC').tz_localize (None )

storm_metadata =relevant_storms [['SID','NAME','datetime_min','datetime_max']].copy ()
storm_metadata =storm_metadata .rename (columns ={'SID':'storm_id','NAME':'storm_name'})
storm_metadata ['storm_start_utc']=storm_metadata ['datetime_min'].apply (lambda x :to_utc_naive (x ).floor ('h'))
storm_metadata ['storm_end_utc']=storm_metadata ['datetime_max'].apply (lambda x :to_utc_naive (x ).ceil ('h'))

storm_metadata ['window_start_utc']=storm_metadata ['storm_start_utc']
storm_metadata ['window_end_utc']=storm_metadata ['storm_end_utc']+pd .to_timedelta (1 ,'d')
storm_metadata ['storm_idx']=np .arange (storm_metadata .shape [0 ],dtype =np .int32 )

county_fips =county_map_df ['FULL_COUNTY_FIPS'].to_numpy ()
county_map_values =county_map_df ['county_map'].to_numpy (dtype =np .int32 )
county_fips_to_row =pd .Series (np .arange (len (county_fips ),dtype =np .int64 ),index =county_fips )
county_fips_set =set (county_fips .tolist ())

peaks_by_county_storm =np .full ((len (county_fips ),storm_metadata .shape [0 ]),-1.0 ,dtype =np .float32 )

storms_by_year =defaultdict (list )
for storm_idx ,row in storm_metadata .iterrows ():
    for year in range (row ['window_start_utc'].year ,row ['window_end_utc'].year +1 ):
        storms_by_year [int (year )].append (int (storm_idx ))

print ('Number of relevant storms:',storm_metadata .shape [0 ])
print ('Number of counties in region:',len (county_fips ))
print ('Years with relevant storms:',sorted (storms_by_year ))


In [ ]:
eaglei_files =sorted (
p for p in eaglei_outage_directory .glob ('*.csv')
if p .name .startswith ('eaglei_outages')
)
if limit_year_files is not None :
    eaglei_files =eaglei_files [:int (limit_year_files )]

for eaglei_file in tqdm (eaglei_files ,desc ='EAGLE-I yearly files'):
    match =re .search (r'(\d{4})',eaglei_file .stem )
    if match is None :
        continue

    file_year =int (match .group (1 ))
    storm_indices_for_this_year =storms_by_year .get (file_year ,[])
    if not storm_indices_for_this_year :
        continue

    chunk_iter =pd .read_csv (
    eaglei_file ,
    usecols =['fips_code','sum','run_start_time'],
    dtype ={'fips_code':str ,'sum':str ,'run_start_time':str },
    chunksize =chunk_size ,
    )

    for chunk in tqdm (chunk_iter ,leave =False ,desc =eaglei_file .name ):
        if chunk .empty :
            continue

        chunk ['fips_code']=chunk ['fips_code'].astype (str ).str .zfill (5 )
        chunk =chunk [chunk ['fips_code'].isin (county_fips_set )].copy ()
        if chunk .empty :
            continue

        chunk ['run_start_time']=pd .to_datetime (chunk ['run_start_time'],utc =True ,errors ='coerce')
        chunk ['sum']=pd .to_numeric (chunk ['sum'],errors ='coerce')
        chunk =chunk .dropna (subset =['run_start_time','sum'])
        if chunk .empty :
            continue

        chunk ['run_start_time']=chunk ['run_start_time'].dt .tz_convert ('UTC').dt .tz_localize (None )

        for storm_idx in storm_indices_for_this_year :
            storm_row =storm_metadata .iloc [storm_idx ]
            start_dt =storm_row ['window_start_utc']
            end_dt =storm_row ['window_end_utc']

            mask =(chunk ['run_start_time']>=start_dt )&(chunk ['run_start_time']<=end_dt )
            if not mask .any ():
                continue

            chunk_for_storm =chunk .loc [mask ,['fips_code','sum']]
            grouped =chunk_for_storm .groupby ('fips_code',sort =False )['sum'].max ()
            if grouped .empty :
                continue

            row_positions =county_fips_to_row .loc [grouped .index ].to_numpy (dtype =np .int64 )
            new_values =grouped .to_numpy (dtype =np .float32 )
            current_values =peaks_by_county_storm [row_positions ,storm_idx ]
            peaks_by_county_storm [row_positions ,storm_idx ]=np .maximum (current_values ,new_values )

peaks_by_county_storm_ge4h =build_peak_customers_affected_ge4h_from_exact_outages (
storm_metadata =storm_metadata ,
county_fips =county_fips ,
county_fips_to_row =county_fips_to_row ,
county_shapefile_path =county_shapefile_path ,
exact_outage_base_directory =exact_outage_base_directory ,
exact_outage_utilities =exact_outage_utilities ,
duration_hours =minimum_duration_hours_for_ge4h_peak ,
)


In [ ]:
storm_output_template =storm_metadata [
['storm_idx','storm_id','storm_name','datetime_min','datetime_max','window_start_utc','window_end_utc']
].copy ()
storm_output_template =storm_output_template .rename (columns ={
'datetime_min':'track_datetime_min',
'datetime_max':'track_datetime_max',
})

for county_row_idx ,(county_fips_code ,county_map_value )in tqdm (
enumerate (zip (county_fips ,county_map_values )),
total =len (county_fips ),
desc ='Writing county files'
):
    county_output =storm_output_template .copy ()
    county_output ['FULL_COUNTY_FIPS']=county_fips_code
    county_output ['county_map']=county_map_value
    county_output ['peak_customers_affected']=peaks_by_county_storm [county_row_idx ,:].astype (np .float32 )
    county_output ['peak_customers_affected_ge4h']=peaks_by_county_storm_ge4h [county_row_idx ,:].astype (np .float32 )

    county_file_path =processed_eaglei_output_directory /f'county_{county_fips_code}.feather'
    county_output .to_feather (county_file_path )


In [ ]:
output_files =sorted (processed_eaglei_output_directory .glob ('county_*.feather'))
print ('County files written:',len (output_files ))

num_non_missing =int (np .sum (peaks_by_county_storm >=0 ))
num_non_missing_ge4h =int (np .sum (peaks_by_county_storm_ge4h >=0 ))
num_total =int (peaks_by_county_storm .size )
print ('Non-missing county-storm peaks:',num_non_missing ,'/',num_total )
print ('Non-missing county-storm peaks (>=4h exact outages):',num_non_missing_ge4h ,'/',num_total )

if output_files :
    sample_path =output_files [0 ]
    sample_df =pd .read_feather (sample_path )
    print ('Sample file:',sample_path .name )
    print (sample_df .head (10 ).to_string (index =False ))
